# 🪐 ExoWeave: The Master Planetary Coupler

Welcome to **ExoWeave**, the overarching orchestration framework that seamlessly stitches together the **ExoWrap** atmospheric model (ExoREM) and the **FuzzyCore** interior model. 

Solving a whole-planet structure is a complex boundary-value problem. The atmosphere dictates how heat escapes, while the interior dictates how much total mass is pulling that atmosphere downward. `exoweave` acts as the master controller, dynamically linking these two domains, matching their chemistries, and iteratively solving for the exact surface gravity required to hold the specified planetary mass together.


---

## 1. The Physics and Mathematics of Coupling

The `ExoCoupler` does not just blindly stack two models on top of each other. It enforces strict physical and thermodynamic continuity across the boundary.

### 1.1 Total Mass Stitching
To verify if a model is physically consistent, `exoweave` integrates the entire atmospheric envelope shell-by-shell from the linking pressure down to the top of the atmosphere. The thickness and mass of each shell are calculated as:

$$dr = -dp \frac{r^2}{\rho G m}$$
$$dm = 4\pi r^2 \rho dr$$

The total calculated mass is the sum of the deep interior mass and this atmospheric mass envelope:

$$M_{calc} = M_{interior} + \sum dm_{atm}$$

### 1.2 The Secant Root-Finder
If $M_{calc}$ does not match the user's target mass, the coupler must adjust the surface gravity ($g$). It uses a **Secant Method** to iteratively hunt for the correct gravity based on the mass error ($e$):

$$g_{n+1} = g_n - e_n \left( \frac{g_n - g_{n-1}}{e_n - e_{n-1}} \right)$$

### 1.3 Chemical Synchronization (The Atmosphere-Interior Handshake)
ExoREM tracks dozens of individual chemical species (like $H_2O$, $CH_4$, $CO$) and their respective Volume Mixing Ratios (VMRs) across the atmospheric grid. However, the deep interior EOS in `fuzzycore` operates on bulk mass fractions: $X$ (Hydrogen), $Y$ (Helium), and $Z$ (Heavy Elements). 



To ensure thermodynamic continuity at the boundary, `exoweave` executes a strict chemical translation exactly at the linking pressure ($P_{link}$). 

**Step 1: Extract Base VMRs and Molar Mass**
The coupler locates the atmospheric layer closest to $P_{link}$ and extracts the mean molar mass ($\mu$) alongside the raw VMRs for molecular hydrogen ($H_2$), atomic hydrogen ($H$), and helium ($He$). It explicitly targets the stable background gases to avoid double-counting atomic elements locked in molecules.

**Step 2: Convert VMR to Mass Fraction ($X$ and $Y$)**
Using standard molar masses ($M_{H2}$, $M_{H}$, and $M_{He}$ in kg/mol), it converts the volume ratios into absolute mass fractions:

$$X = \frac{VMR_{H_2} \cdot M_{H_2} + VMR_H \cdot M_H}{\mu}$$
$$Y = \frac{VMR_{He} \cdot M_{He}}{\mu}$$

**Step 3: Derive Heavy Element Fraction ($Z$) and EOS Ratio**
Since the total mass fraction must sum to $1.0$, the heavy element mass fraction at the boundary is simply the remainder:

$$Z_{base} = 1.0 - (X + Y)$$

Finally, it calculates the internal Helium-to-Fluid ratio ($Y_{ratio}$) required by the `fuzzycore` EOS mixer:

$$Y_{ratio} = \frac{Y}{X + Y}$$

*Note on Fallbacks:* If the RCE model encounters an error extracting the deep VMRs, `exoweave` falls back to an analytical approximation using the user's input metallicity ($Z \approx 0.015 \cdot 10^{Met}$) and a standard proto-solar helium ratio ($Y_{ratio} = 0.26$) to prevent the run from crashing.

---
## 2. Running a Single Coupled Model

The primary entry point for `exoweave` is the `ExoCoupler` class. It accepts a target parameter dictionary and a configuration dictionary, handles all the bootstrapping (like finding smart initial gravity guesses), runs the iteration loop, and outputs a stitched master profile.

In [1]:
from exoweave import ExoCoupler

# 1. Define the physical parameters of the planet
target_params = {
    "mass": 1.0,               # Jupiter masses
    "T_irr": 1100.0,           # Irradiation temperature (K)
    "T_int": 400.0,            # Intrinsic heat flux (K)
    "Met": 0.0,                # Metallicity (log10 Z/Z_solar)
    "core_mass_earth": 15.0,   # Solid core mass in Earth masses
    "iron_fraction": 0.33,     # Earth-like rock/iron ratio
    "f_sed": 1.0,              # Cloud sedimentation parameter
    "kzz": 8.0,                # Eddy diffusion (log10 cm^2/s)
    "debug": False
}

# 2. Define the numerical configuration for the solver
config = {
    "max_iterations": 15,              
    "mass_convergence_threshold": 0.01, # Target 1% mass accuracy
    "p_link_target_bar": 1000.0,        # Link models at 1000 bar
    "min_p_link_bar": 0.1,              # Safeguard against shallow linking
    "resolution": 50,                   # Base atmospheric resolution
    "target_resolution": 500,           # High-res upgrade upon convergence
    "output_dir": "./outputs/single_run"
}

# 3. Initialize the Orchestrator
print("🚀 Launching ExoWeave Coupler...")
coupler = ExoCoupler(target_params=target_params, config=config)

# 4. Fire the main solver loop!
results = coupler.run()

if results['status'] == 'converged':
    print(f"\n✅ Success! Converged in {results['iterations']} iterations.")
    print(f"Final True T_int: {results['final_params']['T_int']:.1f} K")
    print(f"Final g_1bar: {results['final_params']['g_1bar']:.2f} m/s²")
else:
    print("\n❌ Solver failed to converge.")

INFO: 👢 Bootstrapping initial gravity via 1-bar interior pre-solve...


🚀 Launching ExoWeave Coupler...
--- Loading Raw EOS Tables (From Disk) ---
  > Loading Hydrogen...
  > Loading Helium...
  > Loading Water...
  > Loading Rock...
  > Loading Iron...
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


INFO: ✅ Bootstrap successfully locked initial g_1bar = 34.18 m/s²
INFO: 🌌 Grid Setup: Generated mathematical cold-start prior down to 1000.0 bars.
INFO: 
INFO: 🔄 ITERATION 1/15 | Target Mass: 1.0 M_Jup | g: 34.18 m/s²
INFO: ==================================================
INFO: Starting ExoREM Simulation...
INFO: Generated namelist at /private/var/folders/ld/pl_0zzs158sb8h6_y3mrd0lm0000gp/T/tmpcgytlxd6/input.nml
INFO: Running Fortran backend from /Users/cwilkinson/.exowrap/exorem_source/dist/dist/exorem/bin...
INFO: Parsing results from /private/var/folders/ld/pl_0zzs158sb8h6_y3mrd0lm0000gp/T/tmpcgytlxd6/outputs/exowrap_run.h5...
INFO: Simulation complete.
INFO: 🔗 Dynamic Junction: Anchoring to thickest convective block at P = 57.88 bar
INFO: Calculating Z_base and Y_ratio from atmospheric VMRs at 57.87619883491205 bar.
INFO: 🧪 Chemical Sync: Derived Z_base = 0.0141, Y_ratio = 0.2502 (from X=0.7393, Y=0.2466)
INFO: 📊 Breakdown: Interior Mass = 1.9728 M_Jup
INFO: 📊 Breakdown: Atm Mass

🚀 Upgrading simulation to R=500...
💾 Locked P-T profile saved to: /Users/cwilkinson/Documents/Work/Research/Research/Codes/exoweave/outputs/single_run/high_res_tmp_a53e405c/locked_pt_profile.dat
⚡ Executing 0-iteration forward pass...


INFO: Parsing results from /private/var/folders/ld/pl_0zzs158sb8h6_y3mrd0lm0000gp/T/tmpw_qxekrb/outputs/exowrap_run.h5...
INFO: 💾 Permanently saved HDF5 to: /Users/cwilkinson/Documents/Work/Research/Research/Codes/exoweave/outputs/single_run/high_res_tmp_a53e405c/exorem_Tint331.2864988824951_g34.17720752669209.h5
INFO: Simulation complete.
INFO: 🌟 High-resolution upgrade seamlessly injected!
INFO: Stitching profiles at P_link = 57.88 bar
INFO: 🌟 Computing Mega-Catalog Photometry (Cached)...


...

🚀 Upgrading simulation to R=500...
💾 Locked P-T profile saved to: /Users/cwilkinson/Documents/Work/Research/Research/Codes/exoweave/outputs/single_run/high_res_tmp_4c0d7c23/locked_pt_profile.dat
⚡ Executing 0-iteration forward pass...


INFO: Parsing results from /private/var/folders/ld/pl_0zzs158sb8h6_y3mrd0lm0000gp/T/tmpw1409ma2/outputs/exowrap_run.h5...
INFO: 💾 Permanently saved HDF5 to: /Users/cwilkinson/Documents/Work/Research/Research/Codes/exoweave/outputs/single_run/high_res_tmp_4c0d7c23/exorem_Tint305.3787613219604_g17.63062470498534.h5
INFO: Simulation complete.
INFO: 🌟 High-resolution upgrade seamlessly injected!
INFO: Stitching profiles at P_link = 125.89 bar
INFO: 🌟 Computing Mega-Catalog Photometry (Cached)...


🎉 High-resolution upgrade complete!


INFO: ✅ Successfully cached and computed 101 photometric bands!
INFO: ✅ CONVERGED in 4 iterations!
INFO: 💾 Converged model saved successfully to: /Users/cwilkinson/Documents/Work/Research/Research/Codes/exoweave/outputs/single_run/target/M_1.000_Tirr_1100.0_Tint_305.4_Met_0.00_Core_15.0_fsed_1.0_kzz_8.0.pkl



✅ Success! Converged in 4 iterations.
Final True T_int: 305.4 K
Final g_1bar: 17.63 m/s²


---
## 3. Large-Scale Grid Generation

When building datasets for Machine Learning (like CoolTrack), you need thousands of physically consistent models. `exoweave` provides two powerful scripts in its `scripts/` directory to automate this.

### 3.1 `run_grid.py`: Distributed Randomized Sampling
Instead of a rigid Cartesian grid, `run_grid.py` uses Uniform Random Sampling across predefined parameter bounds. This prevents ML models from memorizing discrete steps and forces them to learn true physical gradients.


It uses Python's `ProcessPoolExecutor` to parallelize the runs across all available CPU cores. Crucially, it features a **Pre-Flight Cache Scanner**. By setting a fixed `RANDOM_SEED`, the script can scan the output directory, identify which random models have already finished, and seamlessly resume an interrupted job without losing progress.

In [2]:
# To run the grid generator, you execute the script directly from your terminal.
# Inside scripts/run_grid.py, you can configure your bounds:

"""
PARAM_BOUNDS = {
    "mass": (0.1, 13.0),             # Jupiter Masses
    "T_int": (200.0, 1500.0),        # Internal Temperatures (K)
    "T_irr": (100.0, 1500.0),        # Irradiation (K)
    "Met": (-2, 2),                  # Metallicity
    "core_mass_earth": (1, 100),     # Solid core mass
    "f_sed": (1.0, 7.0),             # Cloud sedimentation
    "kzz": (0.0, 12.0)               # Eddy diffusion
}
TOTAL_RANDOM_MODELS = 1000
"""

# Run it via terminal:
# $ python scripts/run_grid.py

'\nPARAM_BOUNDS = {\n    "mass": (0.1, 13.0),             # Jupiter Masses\n    "T_int": (200.0, 1500.0),        # Internal Temperatures (K)\n    "T_irr": (100.0, 1500.0),        # Irradiation (K)\n    "Met": (-2, 2),                  # Metallicity\n    "core_mass_earth": (1, 100),     # Solid core mass\n    "f_sed": (1.0, 7.0),             # Cloud sedimentation\n    "kzz": (0.0, 12.0)               # Eddy diffusion\n}\nTOTAL_RANDOM_MODELS = 1000\n'

### 3.2 `compile_grid.py`: Assembling the ML Dataset
Once your massive grid of `.pkl` files has finished running, you need to condense it into a machine-readable format. `compile_grid.py` crawls through your output directory and builds two master files:

1. **`master_grid_catalog.csv`**: A lightweight, tabular summary of all runs. It extracts key metadata (Status, Target Mass, True $T_{int}$, Iterations) and highly sought-after observables like $R_{1bar}$ (interpolated 1-bar radius).
2. **`master_grid_data.h5`**: A highly compressed, hierarchical HDF5 binary store. For every successful model, it creates 5 nested folders:
   * `/parameters`: The physical dials.
   * `/stitched_profile`: The continuous top-to-bottom $P-T-\rho$ arrays.
   * `/atmosphere_raw`: The raw ExoREM outputs (VMRs, opacities).
   * `/interior_raw`: The raw fuzzycore outputs (mass tracking, cooling rates).
   * `/photometry`: Synthetic JWST/HST/VLT photometric fluxes.

Like the runner script, `compile_grid.py` is resumable. It opens the HDF5 file in append mode (`'a'`) and checks the existing CSV so it only compiles newly generated models.

In [4]:
# To compile your finished grid into the CSV and HDF5 files, 
# simply run the compiler script from your terminal:

# $ python scripts/compile_grid.py

# This will generate:
# - outputs/master_grid_catalog.csv
# - outputs/master_grid_data.h5

# You can then easily load the catalog in pandas to inspect your dataset:
import pandas as pd

try:
    df_catalog = pd.read_csv("../outputs/master_grid_catalog.csv")
    print(f"Successfully compiled {len(df_catalog)} planetary models!")
    display(df_catalog.head())
except FileNotFoundError:
    print("Run the compile script first to generate the catalog.")

Successfully compiled 4390 planetary models!


,model_id,status,target_mass_Mjup,R_1bar_Rjup,true_mass_Mjup,T_int_dial_K,T_int_true_K,T_irr_K,metallicity,core_mass_Me,...,GAIA_GAIA3.G_flux_Wm2um,GAIA_GAIA3.G_flux_Jy,GAIA_GAIA3.Gbp_flux_Wm2um,GAIA_GAIA3.Gbp_flux_Jy,GAIA_GAIA3.Grp_flux_Wm2um,GAIA_GAIA3.Grp_flux_Jy,TESS_TESS.Red_flux_Wm2um,TESS_TESS.Red_flux_Jy,Kepler_Kepler.K_flux_Wm2um,Kepler_Kepler.K_flux_Jy
0,model_00000,max_iterations_reached,0.5,1.384819,0.455828,400.0,284.282987,1000.0,0.0,10.0,...,1725.184389,2.350160e+14,2167.946054,1.942713e+14,1910.446394,3.902019e+14,2739.752502,5.808507e+14,1302.097432,1.788406e+14
1,model_00001,max_iterations_reached,0.5,1.384819,0.455828,400.0,284.282987,1000.0,0.0,10.0,...,1725.184389,2.350160e+14,2167.946054,1.942713e+14,1910.446394,3.902019e+14,2739.752502,5.808507e+14,1302.097432,1.788406e+14
2,model_00002,max_iterations_reached,0.5,1.384819,0.455828,400.0,284.282987,1000.0,0.0,10.0,...,1725.184389,2.350160e+14,2167.946054,1.942713e+14,1910.446394,3.902019e+14,2739.752502,5.808507e+14,1302.097432,1.788406e+14
3,model_00003,max_iterations_reached,0.5,1.384819,0.455828,400.0,284.282987,1000.0,0.0,10.0,...,1725.184389,2.350160e+14,2167.946054,1.942713e+14,1910.446394,3.902019e+14,2739.752502,5.808507e+14,1302.097432,1.788406e+14
4,model_00004,max_iterations_reached,0.5,1.342318,0.459586,400.0,287.259083,1000.0,0.0,15.0,...,1714.845995,2.336076e+14,2125.495443,1.904673e+14,1937.697541,3.957678e+14,2792.892336,5.921167e+14,1284.833089,1.764694e+14
